# Notebook 06 — Graph Neural Network Models

Trains GraphSAGE and GAT on the PyG logistics graph.
Visualises GAT attention weights to interpret which hub neighbours matter most.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from src.graph.builder import load_graph, graph_to_dataframe
from src.graph.node2vec_emb import load_embeddings, get_node_feature_matrix
from src.models.gnn_models import build_pyg_data, GraphSAGEModel, GATModel, train_gnn
print(f'PyTorch: {torch.__version__}')
try:
    import torch_geometric; print(f'PyG: {torch_geometric.__version__}')
except ImportError:
    print('PyTorch Geometric not installed. Install with: pip install torch-geometric')

In [ ]:
G = load_graph()
embeddings = load_embeddings()
X, node_order = get_node_feature_matrix(G, embeddings)
edge_df = graph_to_dataframe(G)
print(f'Node feature matrix: {X.shape}')
print(f'Edge dataframe: {edge_df.shape}')

In [ ]:
# Edge features for GNN
EDGE_FEAT_COLS = ['osrm_distance', 'osrm_time', 'median_delay_ratio', 'volume', 'src_avg_dwell']
available_edge = [c for c in EDGE_FEAT_COLS if c in edge_df.columns]
data = build_pyg_data(X, node_order, edge_df, available_edge)
print(f'PyG Data: {data}')

In [ ]:
# Train GraphSAGE
from src.models.gnn_models import GraphSAGEModel, train_gnn
sage = GraphSAGEModel(node_feat_dim=X.shape[1], edge_feat_dim=len(available_edge))
sage_result = train_gnn(sage, data, epochs=50, model_name='GraphSAGE')
print('GraphSAGE result:', sage_result)

In [ ]:
# Train GAT
from src.models.gnn_models import GATModel
gat = GATModel(node_feat_dim=X.shape[1], edge_feat_dim=len(available_edge))
gat_result = train_gnn(gat, data, epochs=50, model_name='GAT')
print('GAT result:', gat_result)

In [ ]:
# Training curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0e1117')
for result, label in [(sage_result, 'GraphSAGE'), (gat_result, 'GAT')]:
    hist = result.get('history', {})
    if hist:
        axes[0].plot(hist['train_mae'], label=f'{label} train')
        axes[0].plot(hist['val_mae'], label=f'{label} val', linestyle='--')
        axes[1].plot(hist['val_within15'], label=label)
axes[0].set_title('Training Loss (MAE)', color='white'); axes[0].legend()
axes[1].set_title('Validation Within-15% Accuracy', color='white'); axes[1].legend()
plt.tight_layout()
plt.savefig('reports/06_gnn_training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()